# 🎬 Movie Recommendation System
### DSBDA Mini Project | Content-Based Filtering
---
**Objective:** Build a content-based movie recommendation system using:
- **Features:** keywords, cast, genres, director
- **Vectorizer:** CountVectorizer (+ TF-IDF variant)
- **Similarity:** Cosine Similarity

**Dataset:** TMDB 5000 Movies (`movie_dataset.csv`)

## Step 1: Import Libraries

In [ ]:
# Standard libraries
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('✅ All libraries imported successfully!')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

## Step 2: Load the Dataset

In [ ]:
# Load dataset from local data/ folder
DATA_PATH = os.path.join('data', 'movie_dataset.csv')

df = pd.read_csv(DATA_PATH)

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Preview first 5 rows
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Check missing values
missing = df.isnull().sum()
print('Missing Values:\n')
print(missing[missing > 0].to_string())

In [ ]:
# Descriptive statistics
print('Total Movies:', len(df))
print('Unique Directors:', df['director'].nunique())
print('Sample Genres:', df['genres'].dropna().iloc[0])
print('Sample Keywords:', df['keywords'].dropna().iloc[0])

## Step 4: Feature Engineering — Combine Features

In [ ]:
# Select only relevant feature columns
FEATURE_COLS = ['keywords', 'cast', 'genres', 'director']

# Work with a clean copy
df_clean = df[['title'] + FEATURE_COLS].copy()

# Handle missing values — fill with empty string
for col in FEATURE_COLS:
    df_clean[col] = df_clean[col].fillna('')

print('Missing values after filling:')
print(df_clean.isnull().sum())

In [ ]:
# Normalize: lowercase and strip extra spaces
for col in FEATURE_COLS:
    df_clean[col] = df_clean[col].apply(lambda x: ' '.join(str(x).lower().split()))

# Combine all features into ONE text column
df_clean['combined_features'] = df_clean[FEATURE_COLS].apply(
    lambda row: ' '.join(row.values.astype(str)), axis=1
)

# Preview the combined column
print('Sample combined feature:')
print(df_clean['combined_features'].iloc[0])

In [ ]:
df_clean[['title', 'combined_features']].head(3)

## Step 5: CountVectorizer — Build Feature Matrix

In [ ]:
# CountVectorizer converts text into a word-count matrix
cv = CountVectorizer(stop_words='english', max_features=10000)
count_matrix = cv.fit_transform(df_clean['combined_features'])

print('Count Matrix Shape:', count_matrix.shape)
# (num_movies, num_unique_words)

## Step 6: Cosine Similarity

In [ ]:
# Compute cosine similarity between all movie pairs
cosine_sim = cosine_similarity(count_matrix)

print('Similarity Matrix Shape:', cosine_sim.shape)
# (num_movies x num_movies)
print('\nSample similarity scores for movie 0:')
print(cosine_sim[0][:10])

## Step 7: Recommendation Function

In [ ]:
def get_recommendations(movie_name, df, similarity_matrix, top_n=5):
    """
    Return top-N similar movies for a given movie name.
    """
    # Lowercase search
    titles_lower = df['title'].str.lower()
    query = movie_name.strip().lower()

    # Exact match
    matches = titles_lower[titles_lower == query]
    if matches.empty:
        # Partial match fallback
        matches = titles_lower[titles_lower.str.contains(query, na=False)]

    if matches.empty:
        raise ValueError(f"Movie '{movie_name}' not found!")

    movie_idx = matches.index[0]

    # Get similarity scores
    sim_scores = list(enumerate(similarity_matrix[movie_idx]))
    
    # Sort by score (desc), exclude the movie itself
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != movie_idx]

    top_indices = [s[0] for s in sim_scores[:top_n]]
    return df['title'].iloc[top_indices].tolist()

print('✅ Recommendation function defined!')

## Step 8: Test Recommendations

In [ ]:
# Test with 'Avatar'
test_movie = 'Avatar'
recs = get_recommendations(test_movie, df_clean, cosine_sim, top_n=5)
print(f'\n🎬 Top 5 movies similar to "{test_movie}":')
for i, title in enumerate(recs, 1):
    print(f'  {i}. {title}')

In [ ]:
# Test with another movie
test_movie2 = 'The Dark Knight'
recs2 = get_recommendations(test_movie2, df_clean, cosine_sim, top_n=5)
print(f'\n🎬 Top 5 movies similar to "{test_movie2}":')
for i, title in enumerate(recs2, 1):
    print(f'  {i}. {title}')

In [ ]:
# Test the error case
try:
    get_recommendations('NonExistentMovie12345', df_clean, cosine_sim)
except ValueError as e:
    print('Error handled correctly:', e)

## Step 9: TF-IDF Variant (Optional)

In [ ]:
# TF-IDF gives higher weight to rare, informative words
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(df_clean['combined_features'])

cosine_sim_tfidf = cosine_similarity(tfidf_matrix)

print('TF-IDF Matrix Shape:', tfidf_matrix.shape)

# Test recommendations with TF-IDF
recs_tfidf = get_recommendations('Avatar', df_clean, cosine_sim_tfidf, top_n=5)
print('\n🎬 TF-IDF Recommendations for Avatar:')
for i, t in enumerate(recs_tfidf, 1):
    print(f'  {i}. {t}')

## Step 10: Compare CountVectorizer vs TF-IDF

In [ ]:
movie = 'Inception'

cv_recs = get_recommendations(movie, df_clean, cosine_sim, top_n=5)
tfidf_recs = get_recommendations(movie, df_clean, cosine_sim_tfidf, top_n=5)

compare_df = pd.DataFrame({
    'CountVectorizer': cv_recs,
    'TF-IDF': tfidf_recs
})

print(f'\n📊 Comparison for "{movie}":')
print(compare_df.to_string(index=False))

## ✅ Summary

| Step | Action |
|------|--------|
| 1 | Loaded `movie_dataset.csv` with 4800+ movies |
| 2 | Filled missing values with empty strings |
| 3 | Combined `keywords + cast + genres + director` |
| 4 | Vectorized with `CountVectorizer` (and `TF-IDF`) |
| 5 | Computed `cosine_similarity` matrix |
| 6 | Built `get_recommendations()` function |
| 7 | Tested on multiple movies |

### Run the Streamlit App:
```bash
streamlit run app.py
```